# Figure 11: Cumulative distinct valid names, 4-protocol comparison, per-model grid

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.plots.style import apply_paper_style, model_label, model_color, ordered_models
apply_paper_style()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import (
    load_accessible_professions, load_latent_professions, load_multiturn_professions,
    load_multi_output_professions, cumulative_unique_by_sample, try_load, warn_incomplete_coverage,
)

acc = try_load(load_accessible_professions, label="accessible")
lat = try_load(load_latent_professions, expanded=True, label="latent (expanded)")
mt = try_load(load_multiturn_professions, label="multiturn")
mo = try_load(load_multi_output_professions, label="multi_output")

_EMPTY = pd.DataFrame(columns=["model_version", "query", "x", "cum_unique"])
cum_acc = cumulative_unique_by_sample(acc, group_cols=["model_version", "query"], order_col="sample_id") if acc is not None else _EMPTY
cum_lat = cumulative_unique_by_sample(lat, group_cols=["model_version", "query"], order_col="name_rank") if lat is not None else _EMPTY
cum_mt = cumulative_unique_by_sample(mt, group_cols=["model_version", "query"], order_col="name_rank") if mt is not None else _EMPTY
cum_mo = cumulative_unique_by_sample(mo, group_cols=["model_version", "query"], order_col="name_rank") if mo is not None else _EMPTY

protocols = {"Accessible": cum_acc, "Latent": cum_lat, "Multi-turn": cum_mt, "Multi-output": cum_mo}
for name, cum in protocols.items():
    warn_incomplete_coverage(cum, label=name)


In [ ]:
from src.plots.style import MODEL_ORDER

K_STEPS = 10
models = MODEL_ORDER  # force the full 10-model grid; models with no data anywhere render blank
fig, axes = plt.subplots(2, 5, figsize=(19, 7), sharex=True, sharey=True)
empty_panels = []
any_lines = False
for ax, m in zip(axes.flatten(), models):
    panel_has_data = False
    for proto_name, cum in protocols.items():
        msub = cum[cum["model_version"] == m] if not cum.empty else cum
        if msub.empty:
            continue
        # aggregate cum_unique across queries, per step, taking the mean cumulative count reached
        per_step = msub[msub["x"] <= K_STEPS].groupby("x")["cum_unique"].mean()
        if per_step.empty:
            continue
        ax.plot(per_step.index, per_step.values, label=proto_name, linewidth=1.4)
        panel_has_data = True
        any_lines = True
    if not panel_has_data:
        empty_panels.append(m)
        ax.text(0.5, 0.5, "no data", ha="center", va="center", fontsize=8, color="gray", transform=ax.transAxes)
    ax.set_title(model_label(m), fontsize=9)

if empty_panels:
    print(f"No data for model(s): {empty_panels}")
if any_lines:
    axes[0][0].legend(fontsize=7)
else:
    print("No data at all: every panel is empty.")
fig.suptitle("Cumulative unique valid names by step, averaged over prompts")
plt.tight_layout()
plt.show()
